# Analysis of Dispatch vs. Demand
## 1. Introduction

This Notebook investigates how dispatch from models satisfies demand. We'll load various datasets, perform comparisons, and analyze results through visualizations.

## 2. Import Libraries

In [2]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

## 3. Load and Prepare Data

In [3]:
model_inputs_path = '../../model/inputs/'
model_outputs_path = '../../model/outputs/'
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

### 3.1 Timepoints
It is always advisable to load the timepoints from the model to simplify the transformations of many tables when merging.

In [4]:
# Load timepoints
timepoints = pd.read_csv(model_inputs_path+'timepoints.csv')
timepoints['timepoints'] = timepoints['timepoint_id']
timepoints = timepoints.drop(columns=['timeseries','timepoint_id'])

### 3.2.1 Dispatch by Tecnology from Switch
`dispatch_system` will be compared with the system registry on xm, while `dispatch` will be used to compare by technology.

In [5]:
dispatch_sw = pd.read_csv(model_outputs_path+'dispatch.csv')
dispatch_sw = dispatch_sw.groupby(['timestamp','gen_tech']).agg({
    'DispatchGen_MW' : 'sum'
}).reset_index()

dispatch_sw = pd.merge(dispatch_sw, timepoints, on='timestamp', how='inner')
dispatch_sw = dispatch_sw.sort_values(by='timepoints')

dispatch_sw.head(3)

,timestamp,gen_tech,DispatchGen_MW,timepoints
120,2023_Q1_labor_0h,Eolica,26.734497,1
123,2023_Q1_labor_0h,Thermal,1570.093737,1
124,2023_Q1_labor_0h,pv_solar,0.000000,1


### 3.2.1 Dispatch by Tecnology from XM

In [6]:
dispatch_xm = pd.read_csv(dema_path+'Melted_Gen_Res.csv')
dispatch_xm['DispatchGen_MW'] = dispatch_xm['GeneReal']
# Add Timepoints
dispatch_xm = pd.merge(dispatch_xm, timepoints, on='timestamp', how='inner')
# Sort by timepoints
dispatch_xm = dispatch_xm.sort_values(by='timepoints')

dispatch_xm.head(3)

,timestamp,Tech,GeneReal,DispatchGen_MW,timepoints
114,2023_Q1_labor_0h,Wind,21.928230,21.928230,1
113,2023_Q1_labor_0h,Thermal,1605.389351,1605.389351,1
112,2023_Q1_labor_0h,Solar,0.000645,0.000645,1


### 3.3.1 Dispatch by System from Switch

In [7]:
dispatch_sys = dispatch_sw.groupby(['timestamp','timepoints']).agg({
    'DispatchGen_MW' : 'sum'
    }).reset_index()

### 3.3.2 Demand by System from Switch

In [8]:
loads = pd.read_csv(model_inputs_path+'loads.csv')
loads = loads.groupby(['timepoints']).agg({
    'zone_demand_mw': 'sum'
}).reset_index()

print(loads.shape)
loads.head(3)

(576, 2)


,timepoints,zone_demand_mw
0,1,8433.925584
1,2,7961.776883
2,3,7689.767922


### 3.3.3 Dispatch and Demand by System from XM

In [9]:
XM = pd.read_csv(dema_path+'Melted_DemaGen_Sys.csv')
# Group by timestamp
XM = XM.groupby(['timestamp']).agg({
    'DemaReal_Sistema': 'mean',
    'DemaCome_Sistema': 'mean',
    'Gene_Sistema': 'mean',
    'GeneIdea_Sistema': 'mean'
}).reset_index()
# Adjust to MW
XM['DemaReal_Sistema'] = XM['DemaReal_Sistema'] / 1000
XM['DemaCome_Sistema'] = XM['DemaCome_Sistema'] / 1000
XM['Gene_Sistema'] = XM['Gene_Sistema'] / 1000
XM['GeneIdea_Sistema'] = XM['GeneIdea_Sistema'] / 1000

print(XM.shape)
XM.head(3)

(192, 5)


,timestamp,DemaReal_Sistema,DemaCome_Sistema,Gene_Sistema,GeneIdea_Sistema
0,2023_Q1_holidays_0h,7915.745849,8076.836295,8076.777236,8076.777236
1,2023_Q1_holidays_10h,7725.791177,7856.700819,7856.700819,7856.700819
2,2023_Q1_holidays_11h,8022.721860,8154.997578,8154.919696,8154.919696


## 3.4 System Analysis

### 3.4.1 Is Demand on Switch met by Switch Dispatch?

In [10]:
# Merge tables, result will be: timepoints, timestamp, zone_demand_mw, dispatch_wide 
demand_x_dispatch = pd.merge(loads, dispatch_sys, on='timepoints', how='inner')
# Sort by Timepoints
demand_x_dispatch = demand_x_dispatch.sort_values(by='timepoints')
# Rename columns for better undestanding
demand_x_dispatch.rename(columns={'DispatchGen_MW': 'Dispatch', 'zone_demand_mw': 'Demand'}, inplace=True)

fig = px.line(
    demand_x_dispatch,
    x='timestamp',
    y=['Dispatch', 'Demand'],
    color_discrete_sequence=[colors[0], colors[1]],
    labels={'value': 'Value (MW)', 'variable': 'Series'},
    height=9*50, width=16*50,
    template='plotly_white')
    
fig.show()

### 3.4.2 Are Demand and Dispatch on XM similar to Switch Dispatch?

In [11]:
# Merge tables
demand_x_dispatch = pd.merge(XM, dispatch_sys, on='timestamp', how='inner')
import plotly.express as px
# Sort by timepoints to sort timestamps as well
demand_x_dispatch = demand_x_dispatch.sort_values(by='timepoints')
fig = px.line(
    demand_x_dispatch,
    x='timestamp',
    y=['DemaReal_Sistema', 'DemaCome_Sistema', 'Gene_Sistema', 'GeneIdea_Sistema', 'DispatchGen_MW'],
    color_discrete_sequence=[colors[0], colors[1], colors[2], colors[3], colors[4]],
    labels={'value': 'Values','variable': 'Series'},
    height=9*50, width=16*50,
    template='plotly_white',
    title='Demand/Dispatch XM vs Dispatch Switch'
)
fig.show()

### 3.4.3 Dispatch by Technology XM

In [12]:
gen_res = dispatch_xm.copy()
# Replace prefix 'labor_' by 'L_' and 'holidays_' by 'H_'
gen_res['timestamp'] = gen_res['timestamp'].str.replace('labor_', 'L_')
gen_res['timestamp'] = gen_res['timestamp'].str.replace('holidays_', 'H_')

import plotly.express as px
fig = px.line(
    gen_res,
    x='timestamp', y='DispatchGen_MW',
    color='Tech', title='Generation by Technology (XM)',
    color_discrete_map=tech_colors,
    category_orders={"Tech": tech_order},
    labels={'DispatchGen_MW': 'Reported Generation (MW)'},
    height=9*50, width=16*50,
    template='plotly_white',
)
fig.update_xaxes(dtick=6)
fig.show()

### 3.4.4 Dispatch by Technology Switch

In [13]:
gen_disp = dispatch_sw.copy()
# Add Timepoints
gen_disp = pd.merge(gen_disp, timepoints, on='timestamp', how='inner')
# Sort by timepoints to sort timestamps as well
gen_disp = gen_disp.sort_values(by='timepoints_x')
gen_disp.rename(columns={'timepoints_x': 'timepoints'}, inplace=True)

# Change values on gen_tech to match Switch output
gen_disp['gen_tech'] = gen_disp['gen_tech'].replace(parse_tech)
# Replace prefix 'labor_' by 'L_' and 'holidays_' by 'H_'
gen_disp['timestamp'] = gen_disp['timestamp'].str.replace('labor_', 'L_')
gen_disp['timestamp'] = gen_disp['timestamp'].str.replace('holidays_', 'H_')

gen_disp['Tech'] = gen_disp['gen_tech']
fig = px.line(
    gen_disp, 
    x='timestamp', y='DispatchGen_MW', 
    color='Tech', color_discrete_map=tech_colors,category_orders={"Tech": tech_order},
    title='Generation by Technology (Switch)',
    height=9*50, width=16*50,
    template = 'plotly_white'
)
fig.update_xaxes(dtick=6)
fig.show()

In [14]:
# Load dispatch from transmission
dispatch_tx = pd.read_csv(model_outputs_path+'DispatchTx.csv')
dispatch_tx = dispatch_tx.groupby(['TRANS_TIMEPOINTS_1','TRANS_TIMEPOINTS_2']).agg({
    'DispatchTx' : 'sum' }).reset_index()

# Load transmission_dispatch from plants
dispatch_tx_zn = dispatch_tx.groupby(['TRANS_TIMEPOINTS_1']).agg({
    'DispatchTx' : 'sum' }).reset_index()
# Load dispatch from plant
dispatch_pl = pd.read_csv(model_outputs_path+'dispatch.csv')
dispatch_pl = dispatch_pl.groupby(['generation_project']).agg({
    'DispatchGen_MW' : 'sum' }).reset_index()
# Get zone
pl_zone = pd.read_csv(model_inputs_path+'gen_info.csv')
pl_zone = pl_zone[['GENERATION_PROJECT', 'gen_load_zone']]
dispatch_pl = pd.merge(dispatch_pl, pl_zone,
                       left_on='generation_project', right_on='GENERATION_PROJECT', how='inner')
dispatch_pl = dispatch_pl.groupby(['gen_load_zone']).agg({
    'DispatchGen_MW' : 'sum' }).reset_index()

# Substract transmission from total to get own dispatch
dispatch_pl = pd.merge(dispatch_pl, dispatch_tx_zn,
                       left_on='gen_load_zone', right_on='TRANS_TIMEPOINTS_1', how='inner')
dispatch_pl['DispatchTx'] = dispatch_pl['DispatchGen_MW'] - dispatch_pl['DispatchTx']
dispatch_pl['TRANS_TIMEPOINTS_2'] = dispatch_pl['TRANS_TIMEPOINTS_1']

dispatch_pl = dispatch_pl[['TRANS_TIMEPOINTS_1','TRANS_TIMEPOINTS_2', 'DispatchTx']]
# Concat inner dispatch
dispatch_tx = pd.concat([dispatch_tx, dispatch_pl])

# Get indices from Zones for plot
labels = pd.read_csv(model_inputs_path+'load_zones.csv')['LOAD_ZONE'].tolist()

# Transform TRANS_TIMEPOINTS_1 column into indices
dispatch_tx['TRANS_TIMEPOINTS_1'] = pd.Categorical(
    dispatch_tx['TRANS_TIMEPOINTS_1'], categories=labels).codes
# Transform TRANS_TIMEPOINTS_2 column into indices
dispatch_tx['TRANS_TIMEPOINTS_2'] = len(labels) + pd.Categorical(
    dispatch_tx['TRANS_TIMEPOINTS_2'], categories=labels).codes

labels.extend(labels)

import plotly.graph_objects as go
# Extract columns from the DataFrame
source = dispatch_tx["TRANS_TIMEPOINTS_1"].tolist()
target = dispatch_tx["TRANS_TIMEPOINTS_2"].tolist()
value = dispatch_tx["DispatchTx"].tolist()

# Define colors for nodes
node_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
node_colors.extend(node_colors)

# Function to generate soft random colors
import random
def generate_color():
    r = random.randint(150, 255)  # Red component (light)
    g = random.randint(150, 255)  # Green component (light)
    b = random.randint(150, 255)  # Blue component (light)
    a = 0.6
    return f"rgba({r}, {g}, {b}, {a})"

# Generate soft random colors for links
link_colors = [generate_color() for _ in range(len(source))]

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,  # Space between nodes
        thickness=20,  # Thickness of nodes
        line=dict(color="black", width=0.5),
        label=labels, color=node_colors
    ),
    link=dict(
        source=source,  # Indices of the source nodes
        target=target,  # Indices of the target nodes
        value=value,    # Values of the links
        color=link_colors  # Assign colors to links
    ))])

# Add annotations and styling
fig.update_layout(
    annotations=[
        dict(
            x=0, y=1.1,
            xref="paper", yref="paper", text="Sources",
            showarrow=False, font=dict(size=14, color="black")
        ),
        dict(
            x=1, y=1.1,
            xref="paper", yref="paper", text="Targets",
            showarrow=False, font=dict(size=14, color="black")
        )],        
    height=9*50, width=16*50,
    template="plotly_white"
)

# Show the figure
fig.show()

In [45]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Load and preprocess transmission dispatch data
dispatch_tx = pd.read_csv(model_outputs_path + 'DispatchTx.csv')
dispatch_tx = dispatch_tx.merge(timepoints, left_on='TRANS_TIMEPOINTS_3', right_on='timepoints', how='inner')
dispatch_tx = dispatch_tx[['TRANS_TIMEPOINTS_1', 'TRANS_TIMEPOINTS_2', 'timestamp', 'DispatchTx']]

dispatch_tx_zn = dispatch_tx.groupby(['TRANS_TIMEPOINTS_1', 'timestamp'])['DispatchTx'].sum().reset_index()
dispatch_tx_zn['merge'] = dispatch_tx_zn['TRANS_TIMEPOINTS_1'] + dispatch_tx_zn['timestamp']
dispatch_tx_zn.drop(columns=['timestamp'], inplace=True)

# Load and preprocess plant dispatch data
dispatch_pl = pd.read_csv(model_outputs_path + 'dispatch.csv')
dispatch_pl = dispatch_pl.groupby(['gen_load_zone', 'timestamp'])['DispatchGen_MW'].sum().reset_index()
dispatch_pl['merge'] = dispatch_pl['gen_load_zone'] + dispatch_pl['timestamp']

dispatch_pl = dispatch_pl.merge(dispatch_tx_zn, on='merge', how='inner')
dispatch_pl['DispatchTx'] = dispatch_pl['DispatchGen_MW'] - dispatch_pl['DispatchTx']
dispatch_pl['TRANS_TIMEPOINTS_1'] = dispatch_pl['gen_load_zone']
dispatch_pl['TRANS_TIMEPOINTS_2'] = dispatch_pl['gen_load_zone']
dispatch_pl = dispatch_pl[['TRANS_TIMEPOINTS_1', 'TRANS_TIMEPOINTS_2', 'timestamp', 'DispatchTx']]

dispatch_tx = pd.concat([dispatch_tx, dispatch_pl])

# Parse custom timestamps
def parse_custom_timestamp(ts):
    try:
        year, quarter, hour = int(ts.split('_')[0]), int(ts.split('_')[1][1:]), int(ts.split('_')[-1].replace('h', ''))
        return pd.Timestamp(year=year, month=(quarter - 1) * 3 + 1, day=1, hour=hour)
    except:
        return pd.NaT

dispatch_tx['parsed_timestamp'] = dispatch_tx['timestamp'].apply(parse_custom_timestamp)
dispatch_tx.dropna(subset=['parsed_timestamp'], inplace=True)

dispatch_tx = dispatch_tx.groupby(['TRANS_TIMEPOINTS_1', 'TRANS_TIMEPOINTS_2', 'parsed_timestamp'])['DispatchTx'].sum().reset_index()

# Create widgets for filtering
unique_dates = sorted(dispatch_tx['parsed_timestamp'].dt.to_period('M').unique())
unique_hours = sorted(dispatch_tx['parsed_timestamp'].dt.hour.unique())

date_dropdown = widgets.Dropdown(options=[str(d) for d in unique_dates], description='Year-Month:')
hour_dropdown = widgets.Dropdown(options=unique_hours, description='Hour:')
run_button = widgets.Button(description="Update Sankey")

# Display widgets
display(date_dropdown, hour_dropdown, run_button)

# Define the button click callback
def on_button_click(b):
    date_str = date_dropdown.value
    hour = hour_dropdown.value
    print("saw")
    
    selected_date = pd.Period(date_str)
    filtered_df = dispatch_tx[(dispatch_tx['parsed_timestamp'].dt.to_period('M') == selected_date) &
                              (dispatch_tx['parsed_timestamp'].dt.hour == hour)].copy()
    if filtered_df.empty:
        print(f"No data available for {date_str} at hour {hour}")
        return
    
    labels = pd.read_csv(model_inputs_path + 'load_zones.csv')['LOAD_ZONE'].tolist()
    filtered_df['TRANS_TIMEPOINTS_1'] = pd.Categorical(filtered_df['TRANS_TIMEPOINTS_1'], categories=labels).codes
    filtered_df['TRANS_TIMEPOINTS_2'] = len(labels) + pd.Categorical(filtered_df['TRANS_TIMEPOINTS_2'], categories=labels).codes
    labels.extend(labels)

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, color=node_colors, line=dict(color='black', width=0.5), label=labels),
        link=dict(source=filtered_df['TRANS_TIMEPOINTS_1'], target=filtered_df['TRANS_TIMEPOINTS_2'],
                  value=filtered_df['DispatchTx'], color=[generate_color() for _ in range(len(filtered_df))])
    )])

    fig.update_layout(title_text=f"Transmission Dispatch for {date_str} at {hour:02d}:00",
                      height=9*50, width=16*50, template="plotly_white")
    fig.show()

# Connect the button to the function
run_button.on_click(on_button_click)

Dropdown(description='Year-Month:', options=('2023-01', '2023-04', '2023-07', '2023-10'), value='2023-01')

Dropdown(description='Hour:', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2…

Button(description='Update Sankey', style=ButtonStyle())

### 3.4.5 Dispatch by Technology Switch x XM

In [16]:
# Take both Sets
gen_res = gen_res[['timestamp', 'Tech', 'DispatchGen_MW', ]]
gen_disp = gen_disp[gen_disp['timestamp'].str.startswith('2023')][['timestamp', 'Tech', 'DispatchGen_MW']]

# Crete individual plots
fig1 = px.line(
    gen_res, x='timestamp', y='DispatchGen_MW', color='Tech', title='Generation Reported (XM)',
    color_discrete_map=tech_colors, category_orders={"Tech": tech_order})
fig2 = px.line(
    gen_disp, x='timestamp', y='DispatchGen_MW', color='Tech', title='Generation Predicted (Switch)',
    color_discrete_map=tech_colors, category_orders={"Tech": tech_order})

# Create subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("Generation Reported (XM)", "Generation Predicted (Switch)"))
# Add figures to subplots
for trace in fig1['data']: fig.add_trace(trace, row=1, col=1)
for trace in fig2['data']: fig.add_trace(trace, row=1, col=2)

# Only show one legend
names = set()
for trace in fig['data']:
    if (trace.name in names): trace.showlegend = False
    else: names.add(trace.name)

# Add buttons to change y-range
fig.update_layout(
    title='Base Year (2023)',
    updatemenus=[
        go.layout.Updatemenu(
            buttons=list([
                dict(args=[{"yaxis.autorange": True, "yaxis2.autorange": True}], label="Auto", method="relayout"),
                dict(args=[{"yaxis.range": [5, 30], "yaxis2.range": [5, 30]}], label="30 Fixed", method="relayout"),
                dict(args=[{"yaxis.range": [0, 700], "yaxis2.range": [0, 700]}], label="700 Fixed", method="relayout"),
                dict(args=[{"yaxis.range": [0, 9000], "yaxis2.range": [0, 9000]}], label="9k Fixed", method="relayout")]),
            direction="down", x=1.2, xanchor="right", y=1.2
        )],
    height=9*50, width=16*50,
    template="plotly_white")

fig.update_xaxes(dtick=12, row=1, col=1)
fig.update_xaxes(dtick=12, row=1, col=2)
fig.show()

In [17]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_q1_generation(gen_res, gen_disp, tech_colors, tech_order):   
    # Exctract Quarter, daytime, hour
    for df in [gen_res, gen_disp]:
        df['Quarter'] = df['timestamp'].str.extract(r'Q([1-4])')[0].apply(lambda x: f'Q{x}')
        df['DayType'] = df['timestamp'].str.extract(r'_(L|H)_')[0].map({'L': 'Labor', 'H': 'Holiday'})
        df['Hour'] = df['timestamp'].str.extract(r'_(\d{1,2})h$')[0].astype(int)
    
    daytypes = ['Labor', 'Holiday']
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            "Q1 Labor - Reported (UPME)", "Q1 Holiday - Reported (UPME)",
            "Q1 Labor - Simulated (Switch)", "Q1 Holiday - Reportado Simulated (Switch)"],
        shared_yaxes=True
    )    
    # Map
    position_map = {
        ('Labor', 'Pred'): (1, 1),
        ('Holiday', 'Pred'): (1, 2),
        ('Labor', 'Real'): (2, 1),
        ('Holiday', 'Real'): (2, 2),
    }
    
    for daytype in daytypes:
        # Predicated Data
        disp_df = gen_disp[(gen_disp['DayType'] == daytype) & (gen_disp['Quarter'] == 'Q1')]
        fig_pred = px.line(
            disp_df, x='Hour', y='DispatchGen_MW', color='Tech',
            color_discrete_map=tech_colors, category_orders={"Tech": tech_order}
        )
        for trace in fig_pred['data']:
            #trace.update(fill='tozeroy')
            fig.add_trace(trace, row=position_map[(daytype, 'Pred')][0], col=position_map[(daytype, 'Pred')][1])
        
        # Real data
        res_df = gen_res[(gen_res['DayType'] == daytype) & (gen_res['Quarter'] == 'Q1')]
        fig_real = px.line(
            res_df, x='Hour', y='DispatchGen_MW', color='Tech',
            color_discrete_map=tech_colors, category_orders={"Tech": tech_order}
        )
        for trace in fig_real['data']:
            #trace.update(fill='tozeroy')
            fig.add_trace(trace, row=position_map[(daytype, 'Real')][0], col=position_map[(daytype, 'Real')][1])
    
    # Hide duplicated legends
    names = set()
    for trace in fig['data']:
        if trace.name in names:
            trace.showlegend = False
        else:
            names.add(trace.name)
    
    # Layout final
    fig.update_layout(
        height=12*50, width=16*50,
        template="plotly_white"
    )
    fig.update_xaxes(dtick=3, showticklabels=True, title_text=None)
    fig.update_yaxes(showticklabels=True, title_text=None)
    fig.update_annotations(font=dict(size=11))
    
    fig.show()

plot_q1_generation(gen_res, gen_disp, tech_colors, tech_order)

In [18]:
def plot_q1_generation(gen_res, gen_disp, tech_colors, tech_order):   
    for df in [gen_res, gen_disp]:
        df['Quarter'] = df['timestamp'].str.extract(r'Q([1-4])')[0].apply(lambda x: f'Q{x}')
        df['DayType'] = df['timestamp'].str.extract(r'_(L|H)_')[0].map({'L': 'Labor', 'H': 'Holiday'})
        df['Hour'] = df['timestamp'].str.extract(r'_(\d{1,2})h$')[0]+'h'
    
    daytypes = ['Labor', 'Holiday']
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=["Q1 Labor", "Q1 Holiday", "Q1 Labor", "Q1 Holiday"],
        shared_yaxes=True,
        vertical_spacing=0.22, horizontal_spacing=0.06
    )
    position_map = {
        ('Labor', 'Real'): (1, 1), ('Holiday', 'Real'): (1, 2),
        ('Labor', 'Pred'): (2, 1), ('Holiday', 'Pred'): (2, 2),
    }
    legend_shown = set()
    for daytype in daytypes:
        for tipo, source in [('Real', gen_res), ('Pred', gen_disp)]:
            df_source = source[(source['DayType'] == daytype) & (source['Quarter'] == 'Q1')]
            for tech in tech_order:
                df_tech = df_source[df_source['Tech'] == tech]
                show = tech not in legend_shown
                if show:
                    legend_shown.add(tech)
                fig.add_trace(
                    go.Scatter(
                        x=df_tech['Hour'],
                        y=df_tech['DispatchGen_MW'],
                        mode='lines',
                        name=tech,
                        stackgroup='one',
                        line_shape='linear',
                        legendgroup=tech,
                        showlegend=show,
                        line=dict(color=tech_colors.get(tech, '#333333')),
                        fillcolor=tech_colors.get(tech, '#333333')
                    ),
                    row=position_map[(daytype, tipo)][0],
                    col=position_map[(daytype, tipo)][1]
                )
    # Final Layout
    fig.update_layout(
        height=12*50, width=16*50,
        template="plotly_white",
        legend=dict(
            orientation="h",
            y=-0.1, x=0.5,
            xanchor='center'
        ),
        font_family="Arial", font_size=16
    )
    fig.add_annotation(
    text="Generation [MW]",
    xref="paper", yref="paper",
    x=-0.1, y=0.5,
    showarrow=False,
    textangle=-90)
    
    fig.update_xaxes(dtick=4, showticklabels=True, title_text=None)
    fig.update_yaxes(range=[-400, 12000])
    
    fig.add_annotation(
        text="Actual Generation (XM)",
        xref="paper", yref="paper",
        x=0.5, y=1.12,
        showarrow=False,
        font=dict(size=18, family="Arial"),
        xanchor="center"
    )
    fig.add_annotation(
        text="Simulated Generation (Switch)",
        xref="paper", yref="paper",
        x=0.5, y=0.49,
        showarrow=False,
        font=dict(size=18, family="Arial"),
        xanchor="center"
    )

    fig.write_image(f"../images/Q1 Generation (XM - Switch).png")
    fig.show()

plot_q1_generation(gen_res, gen_disp, tech_colors, ["Hydro", "Run of River", "Thermal", "Wind", "Solar"])

In [44]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Agrupar y preparar los datos reales (XM)
gen_res_grouped = gen_res.groupby(['Quarter', 'DayType', 'Tech'])['DispatchGen_MW'].sum().reset_index()
gen_res_grouped['DispatchGW'] = gen_res_grouped['DispatchGen_MW'] / 1000

# Agrupar y preparar los datos simulados (Switch)
gen_disp_grouped = gen_disp.groupby(['Quarter', 'DayType', 'Tech'])['DispatchGen_MW'].sum().reset_index()
gen_disp_grouped['DispatchGW'] = gen_disp_grouped['DispatchGen_MW'] / 1000

# Crear una columna combinada para el eje X
gen_res_grouped['Category'] = gen_res_grouped['Quarter'] + '-' + gen_res_grouped['DayType']
gen_disp_grouped['Category'] = gen_disp_grouped['Quarter'] + '-' + gen_disp_grouped['DayType']

# Orden consistente
categories = ['Q1-Labor', 'Q1-Holiday', 'Q2-Labor', 'Q2-Holiday', 'Q3-Labor', 'Q3-Holiday', 'Q4-Labor', 'Q4-Holiday']
sources = ['Hydro', 'Run of River', 'Thermal', 'Solar', 'Wind']

# Crear subplots
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Actual Generation (XM)", "Simulated Generation (Switch)"))

threshold = 20 # Giga
# Agregar datos reales (XM)
for source in sources:
    data = gen_res_grouped[gen_res_grouped['Tech'] == source]
    y = [data[data['Category'] == cat]['DispatchGW'].sum() for cat in categories]
    text = [f'{val:.1f}' if val >= threshold else '' for val in y]
    fig.add_trace(go.Bar(
        name=source,
        x=categories,
        y=y,
        text=text,
        texttemplate='%{text}',
        textposition='inside',
        insidetextanchor='middle',
        textangle=0,
        marker_color=tech_colors[source]
    ), row=1, col=1)

# Agregar datos simulados (Switch)
for source in sources:
    data = gen_disp_grouped[gen_disp_grouped['Tech'] == source]
    y = [data[data['Category'] == cat]['DispatchGW'].sum() for cat in categories]
    text = [f'{val:.1f}' if val >= threshold else '' for val in y]
    fig.add_trace(go.Bar(
        name=source,
        x=categories,
        y=y,
        text=text,
        texttemplate='%{text:.1f}', textposition='inside', insidetextanchor='middle', textangle=0,
        marker_color=tech_colors[source],
        showlegend=False  # evitar duplicar leyenda
    ), row=1, col=2)

# Layout general
fig.update_layout(
    barmode='stack',
    title_text='2023 Generation per Typical day',
    yaxis_title='Generation [GWh/day]',
    height=9*50, width=16*50,
    template='plotly_white',
    xaxis_tickangle=-90,
    xaxis2_tickangle=-90,
    legend_traceorder='normal',
    legend=dict(
        orientation="h", y=-0.4, x=0.5, xanchor='center'
    ),
    font=dict(family="Arial", size=14)
)

# Cambiar tamaño de títulos de subplots (accediendo a annotations)
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=18)

fig.write_image(f"../images/Anual Generation 2023 (XM - Switch)).png")
fig.show()


In [21]:
# Align dataframes based on 'timestamp', 'Tech'
gen_res['DispatchGen_MW_xm'] = gen_res['DispatchGen_MW']
gen_disp['DispatchGen_MW_sw'] = gen_disp['DispatchGen_MW']
gen_compare = pd.merge(gen_res, gen_disp, on=['timestamp', 'Tech'])
# Calculate error percentage
gen_compare['percentage_error'] = 100*((gen_compare['DispatchGen_MW_sw'] / gen_compare['DispatchGen_MW_xm']) - 1)
# Remove rows where 'type' contains '_0h' and 'tech' is 'solar'
hours = r'_19h|_20h|_21h|_22h|_23h|_0h|_1h|_2h|_3h|_4h|_5h|_6h|_7h'
condition = ((gen_compare['Tech'] == 'Solar') & (gen_compare['timestamp'].str.contains(hours)))
gen_compare.loc[condition, ['percentage_error']] = np.nan
#print(gen_compare.head(5))

import plotly.express as px
fig = px.line(
    gen_compare,
    x='timestamp', y='percentage_error', 
    color='Tech', color_discrete_map=tech_colors, category_orders={"Tech": tech_order},
    labels={"percentage_error": "Percentage Error [%]", "timestamp":"Timestamp"},
    height=9*50, width=16*50, template="plotly_white")
fig.update_layout(yaxis=dict(range=[-100, 150]))
fig.update_xaxes(dtick=6)
fig.write_image("../images/Percentage Error.png")
fig.show()

### Error by Quartil

In [22]:
# Align dataframes based on 'timestamp', 'Tech'
gen_compare_Q = pd.merge(gen_res, gen_disp, on=['timestamp', 'Tech'])
gen_compare_Q['Quarter'] = gen_compare_Q['timestamp'].apply(lambda x: x[:7])

gen_compare_Q = gen_compare_Q.groupby(['Quarter','Tech']).agg({
    'DispatchGen_MW_xm': 'mean', 'DispatchGen_MW_sw' : 'mean'
}).reset_index()

# Calculate error percentage
gen_compare_Q['percentage_error'] = 100 * ((gen_compare_Q['DispatchGen_MW_sw'] / gen_compare_Q['DispatchGen_MW_xm']) -1)
#print(gen_compare_Q.head(5))
import plotly.express as px
fig = px.line(
    gen_compare_Q, 
    x='Quarter', y='percentage_error', 
    color='Tech', color_discrete_map=tech_colors, category_orders={"Tech": tech_order},
    labels={"percentage_error": "Percentage Error [%]"},
    height=9*50, width=16*50, template="plotly_white")
fig.write_image("../images/Percentage Error Anual.png")
fig.show()

In [23]:
# Alinear los dataframes usando merge
gen_compare_year = pd.merge(gen_res, gen_disp, on=['timestamp', 'Tech'])

gen_compare_year = gen_compare_Q.groupby(['Tech']).agg({
    'DispatchGen_MW_xm': 'mean', 'DispatchGen_MW_sw' : 'mean'
}).reset_index()

# Calcular el porcentaje
gen_compare_year['percentage'] = (gen_compare_year['DispatchGen_MW_sw'] / gen_compare_year['DispatchGen_MW_xm']) - 1
print('Error by Tech (Year)')
gen_compare_year

Error by Tech (Year)


,Tech,DispatchGen_MW_xm,DispatchGen_MW_sw,percentage
0,Hydro,6229.754328,6562.123686,0.053352
1,Run of River,383.952418,441.736690,0.150499
2,Solar,146.755575,192.290016,0.310274
3,Thermal,1925.710945,1636.313360,-0.150281
4,Wind,19.660513,24.543755,0.248378


In [24]:
# Promedio ponderado de GeneReal y percentage
weighted_avg_percentage = \
    (gen_compare_year['DispatchGen_MW_xm'] * gen_compare_year['percentage']).sum() / gen_compare_year['DispatchGen_MW_xm'].sum()
print(f"Average error percentage: {weighted_avg_percentage*100:.6f}%")

Average error percentage: 1.736465%
